当准备轴承故障信号数据时，需要考虑两个方面：
1. 轴承信号数据和对应的标签。
2. 同时，为了构建图结构，还需要确定节点之间的连接关系。

In [1]:
from joblib import dump, load

# 加载数据
train_set = load('train_set') 
val_set = load('val_set') 
test_set = load('test_set') 
print(train_set.shape)  # 二维数组，行代表样本数量， 列代表信号长度和标签
print(val_set.shape)
print(test_set.shape)

(1631, 1025)
(466, 1025)
(233, 1025)


轴承信号数据： 轴承信号数据是分类诊断的主要输入，是一个包含一维序列的列表，其中每一行代表一个轴承的信号。每个序列的长度为1024个数据点和一个对应的分类标签

# 构建图结构：

轴承信号是一维序列，但我们可以通过构建图结构来捕捉序列之间的相对位置关系。使用k近邻方法来建立节点之间的边连接关系。对于每个节点，找到其最近的k个邻居节点，并将它们与该节点连接起来。这样可以形成轴承信号的图结构，其中每个节点对应一个数据点，边表示节点之间的连接关系

In [2]:
# 第一步  导入所需的库和模块：
import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors

# 第二步， 使用k近邻方法，可以使用NearestNeighbors类从信号数据中找到每个节点的最近邻节点
def make_NearestNeighbors(signals, m_k=5):
    '''
        参数 signals: 单个信号
             m_k    :  表示每个节点与其最近的k个邻居连接
        返回 distances: 每个节点与其最近邻节点之间的距离
            indices: 最近邻节点的索引
    '''
    # 创建NearestNeighbors对象
    k = m_k  # 设置k值，表示每个节点与其最近的k个邻居连接
    nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    # 训练NearestNeighbors模型
    nn.fit(signals)
    # 使用模型找到每个节点的最近邻节点
    # distances包含每个节点与其最近邻节点之间的距离，indices包含最近邻节点的索引
    distances, indices = nn.kneighbors(signals)
    return distances, indices

# 第三步， 构建边的连接关系： 根据最近邻节点的索引，可以构建边的连接关系
def get_edge_indexs(signals, indices):
    '''
        参数 signals: 单个信号
             indices: 最近邻节点的索引
        返回 edge_index: 边的连接关系
          
    '''
    edge_index = []
    for i in range(len(signals)):
        for j in indices[i]:
            edge_index.append([i, j])  # 添加节点之间的连接关系

    edge_index = np.array(edge_index).T  # 转置为正确的形状

    # 转换为torch.tensor类型
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    return edge_index

# 创建图数据

使用torch.tensor将轴承信号数据、边的连接关系和标签转换为张量形式。具体地，使用torch.tensor函数将信号数据转换为torch.float类型的张量，边的连接关系转换为torch.long类型的张量，标签转换为torch.long类型的张量。这样就可以创建一个Data对象，其中x属性表示节点特征（轴承信号数据），edge_index属性表示边的连接关系，y属性表示节点的标签

In [3]:
from torch_geometric.data import Data
from joblib import dump, load


def make_graph_dataset(dataframe, m_k):
    '''
        参数 dataframe: 轴承数据
                   m_k: 表示每个节点与其最近的k个邻居连接
        返回 edge_index: 边的连接关系 
    '''
    # 信号值
    x_data = dataframe.iloc[:,0:-1].values
    # x_data = torch.tensor(x_data.values, dtype=torch.float)  # 节点特征，轴承信号数据
    # 标签值
    y_labels = dataframe.iloc[:,-1].values
    # y_labels = torch.tensor(y_labels.values, dtype=torch.long)  # 节点标签，轴承故障类型 类别标签

    # 创建一个包含多个图数据的列表
    dataset = []  
    for index in range(dataframe.shape[0]):
        # 类别标签
        y = y_labels[index]
        # 标签转换为 tensor
        y = torch.tensor(y, dtype=torch.long)  # 节点标签，轴承故障类型
        distances, indices = make_NearestNeighbors(x_data[index].reshape(-1, 1), m_k=5)
        # 边的连接关系
        edge_index = get_edge_indexs(x_data[index], indices)
        # 信号转换为 tensor
        x = torch.tensor(x_data[index].reshape(-1, 1), dtype=torch.float)  # 节点特征，轴承信号数据
        data = Data(x=x, edge_index=edge_index, y=y)
        dataset.append(data)

    print(len(dataset))
    return dataset

# 设置k值，表示每个节点与其最近的k个邻居连接
K = 5
train_dataset = make_graph_dataset(train_set, K)
val_dataset = make_graph_dataset(val_set, K)
test_dataset = make_graph_dataset(test_set, K)

# 保存数据
dump(train_dataset, 'train_dataset')
dump(val_dataset, 'val_dataset')
dump(test_dataset, 'test_dataset')

1631
466
233


['test_dataset']

In [4]:
# 看一个 数据集结构
train_dataset  

[Data(x=[1024, 1], edge_index=[2, 5120], y=0),
 Data(x=[1024, 1], edge_index=[2, 5120], y=7),
 Data(x=[1024, 1], edge_index=[2, 5120], y=5),
 Data(x=[1024, 1], edge_index=[2, 5120], y=6),
 Data(x=[1024, 1], edge_index=[2, 5120], y=4),
 Data(x=[1024, 1], edge_index=[2, 5120], y=6),
 Data(x=[1024, 1], edge_index=[2, 5120], y=4),
 Data(x=[1024, 1], edge_index=[2, 5120], y=8),
 Data(x=[1024, 1], edge_index=[2, 5120], y=0),
 Data(x=[1024, 1], edge_index=[2, 5120], y=0),
 Data(x=[1024, 1], edge_index=[2, 5120], y=7),
 Data(x=[1024, 1], edge_index=[2, 5120], y=1),
 Data(x=[1024, 1], edge_index=[2, 5120], y=4),
 Data(x=[1024, 1], edge_index=[2, 5120], y=9),
 Data(x=[1024, 1], edge_index=[2, 5120], y=1),
 Data(x=[1024, 1], edge_index=[2, 5120], y=7),
 Data(x=[1024, 1], edge_index=[2, 5120], y=0),
 Data(x=[1024, 1], edge_index=[2, 5120], y=1),
 Data(x=[1024, 1], edge_index=[2, 5120], y=8),
 Data(x=[1024, 1], edge_index=[2, 5120], y=2),
 Data(x=[1024, 1], edge_index=[2, 5120], y=0),
 Data(x=[1024